## IMPORTING LIBRARIES

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('data/dataset2.csv')



In [ ]:
pd.set_option("display.max_columns", None)
sns.set_theme(style="darkgrid")

RANDOM_STATE = 42
TARGET_COL = "price"

In [ ]:
print('Dataframe shape : ', df.shape)
print(df.head())


## EXPLORATORY DATA ANALYSIS 

In [ ]:
print(df.columns)
df.info()
print(df.shape)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()

print('Numerical columns : ', num_cols)
print('Categorical columns : ', cat_cols)



### CHECKING MISSING VALUES 

In [ ]:
print('Missing values in each column : ','\n')
print(df.isnull().sum())

### CHECKING DUPLICATES 

In [ ]:
print(df.duplicated().sum())

In [ ]:
print(df.describe())

## CHECKING THE DISTRIBUTION OF TARGET COL 

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df[TARGET_COL], bins=20 ,kde=True)
plt.title('Distribution of Price')
plt.xlabel('Price')
plt.ylabel('Counts')
plt.show()

In [ ]:
print(num_cols)

correlation_matrix = df[num_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
for col in cat_cols:
    print('\n')
    print(col)
    print(df[col].value_counts())

In [ ]:
plt.figure(figsize=(15,10))

for i,col in enumerate(cat_cols,1):
    plt.subplot(3,3,i)
    sns.boxplot(x=col,y=TARGET_COL,data=df)
    plt.title(f'Boxplot of {col} vs {TARGET_COL}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    print(col)
    print(df[col].unique())


In [ ]:
cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

df[cols] = df[cols].replace({'yes':1,'no':0})


df = pd.get_dummies(
    df,
    columns=['furnishingstatus'],
    drop_first=True,
    dtype=int
)

In [ ]:
X = df.drop(columns='price')

y = df[TARGET_COL]

print(X.shape)
print(y.shape)



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)

print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)



In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train) 




In [ ]:
binary_cols = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]

for col in binary_cols:
    df[col] = df[col].astype(int)

In [ ]:
y_pred = lr.predict(X_test)



In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
results = pd.DataFrame({'Actual':y_test,'Predicted':y_pred})

print(results.head(10))

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, y_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')

plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)



In [ ]:
mae_dt = mean_absolute_error(
    y_test,
    y_pred_dt
)

rmse_dt = mean_squared_error(
    y_test,
    y_pred_dt
) ** 0.5

r2_dt = r2_score(
    y_test,
    y_pred_dt
)

print("MAE :", mae_dt)
print("RMSE:", rmse_dt)
print("R2  :", r2_dt)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_rf = mean_squared_error(y_test, y_pred_rf) ** 0.5

r2_rf = r2_score(y_test, y_pred_rf)

print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R2  :", r2_rf)

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [mae, mae_dt, mae_rf],
    'RMSE': [rmse, rmse_dt, rmse_rf],
    'R2': [r2, r2_dt, r2_rf]
}).sort_values('R2', ascending=False)


print(comparison)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_
})

coef_df = coef_df.sort_values(
    by='Coefficient',
    ascending=False
)

print(coef_df)


In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(data=coef_df,x='Coefficient',y='Feature')
plt.title('Feature Coefficients in Linear Regression')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.show()

In [ ]:
import joblib

joblib.dump(lr,'house_price_model.pkl')


In [ ]:
feature_list = X.columns.tolist()
print(feature_list)

In [ ]:
joblib.dump(feature_list, "feature_names.pkl")